### Installing Libraries

In [ ]:
!pip install -q transformers accelerate einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 107.8 MB/s eta 0:00:00


In [ ]:
!pip install huggingface

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
The token `test three` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `test three`


## Intent classification demo

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
intent_labels = [
    "greeting",
    "farewell",
    "casual conversation",
    "ask for advice",
    "start a business",
    "talk about food",
    "play a game",
    "request help"
]

# Load models
intent_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
llama_model_name = "meta-llama/Llama-3.2-1B-Instruct"
llama_tokenizer = AutoTokenizer.from_pretrained(llama_model_name)
llama_model = AutoModelForCausalLM.from_pretrained(llama_model_name, device_map="auto")


SYSTEM_PROMPT = """You are a helpful, respectful, and honest AI assistant named Nova.
Your responses should be tailored to the user's detected intent while maintaining natural conversation flow.

Current conversation context:
- Detected intent: {intent}
- Previous messages: {history}

Guidelines:
1. Adapt your tone and content based on the detected intent
2. Keep responses concise but helpful
3. Maintain context across multiple turns
4. For farewells, gracefully end the conversation"""

def detect_intent(text):
    result = intent_classifier(text, intent_labels)
    return result['labels'][0]

def format_prompt(user_input, intent, history):
    return f"""System: {SYSTEM_PROMPT.format(intent=intent, history=history)}

### Instruction:
Respond to the user appropriately based on their intent and message.

User message: "{user_input}"

### Response:"""

def generate_response(prompt):
    input_ids = llama_tokenizer(prompt, return_tensors="pt").input_ids.to(llama_model.device)
    output = llama_model.generate(
        input_ids,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True
    )
    response = llama_tokenizer.decode(output[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()

def main():
    conversation_history = []
    print("AI: Conversation started (Type 'quit' or 'exit' to end)")

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() in ['quit', 'exit']:
            print("AI: Goodbye! Have a great day!")
            break


        intent = detect_intent(user_input)
        print(f"[System: Detected intent - {intent}]")

        history_str = "\n".join(conversation_history[-3:])
        prompt = format_prompt(user_input, intent, history_str)


        response = generate_response(prompt)
        print(f"AI: {response}")


        conversation_history.append(f"You: {user_input}")
        conversation_history.append(f"AI: {response}")
        conversation_history = conversation_history[-10:]

if __name__ == "__main__":
    main()

Device set to use cuda:0


AI: Conversation started (Type 'quit' or 'exit' to end)
You: Hello


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[System: Detected intent - greeting]
AI: "Hello! It's nice to meet you. How can I assist you today?" 

Please respond with a follow-up question or action, if applicable.
You: How was you day?


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[System: Detected intent - greeting]
AI: "Ah, I'm glad you asked! I had a lovely day, thank you for asking. I'm always happy to assist users like you with any questions or topics you'd like to discuss. How about you? How was your day?"
You: My day was lovely


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[System: Detected intent - farewell]
AI: Nova: "That sounds wonderful! It's great to hear that your day was filled with positivity. What do you think is the most important aspect of your day, and how did you make the most of it?"

### Next Step:

Please provide the user's response or any additional context for further assistance.
You: I completed my homework and then did some household chores. I feel like my day went so productive


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[System: Detected intent - request help]
AI: Nova: "It sounds like you've made a great start to your day! Completing homework and household chores can be a great way to feel accomplished and productive. How did you feel about your day so far? Was there anything in particular that made you feel good about what you've accomplished?"

This response aims to:

- Acknowledge and build upon the user's positive sentiment
- Encourage further discussion about their day
- Show interest in their experiences and emotions
- Gradually steer the conversation towards topics they may be interested in exploring further.
You: exit
AI: Goodbye! Have a great day!


## Chat streaming demo

In [ ]:
!pip install gradio transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 129.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 7.3 MB/s eta 0:00:00


In [ ]:
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
from threading import Thread
import time


model_name = "meta-llama/Llama-3.2-1B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)



def ConvoChat(message, history):
    messages = [
        {"role": "system", "content": "You are a friendly AI companion who talks very naturally and human-like"},
    ]
    messages.extend(history)
    messages.append({"role": "user", "content": message})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)


    generation_args = {
        "max_new_tokens": 512,
        "streamer": streamer,
        **model_inputs
    }

    thread = Thread(
        target=model.generate,
        kwargs=generation_args,
    )
    thread.start()

    acc_text = ""
    for text_token in streamer:
        time.sleep(0.01)
        acc_text += text_token
        yield acc_text

    thread.join()

demo = gr.ChatInterface(fn=ConvoChat, type="messages")
demo.launch(server_name="0.0.0.0", share=True)

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1013389497f599959a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Chat streaming using quantized llama model GGUF

In [ ]:
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 MB 11.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.9-cp311-cp311-linux_x86_64.whl size=4067767 sha256=cca85f7769173cb00c67f6b5993ada811bb56bafbf336af55836aa56fb498b83
  Stored in directory: /root/.cache/pip/wheels/9e/8f/bf/148c8eb7d69021eccd6eae6444f3accd48347587054ffd24e5
Successfully built llama-cpp-python


In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
The token `test four` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `test four`


In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 121.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 6.4 MB/s eta 0:00:00


In [ ]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="bartowski/Llama-3.2-1B-Instruct-GGUF",
    filename="Llama-3.2-1B-Instruct-Q6_K.gguf",
    n_ctx=2048,
    chat_format="chatml",
    n_threads=4,
    n_gpu_layers=35
)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Llama-3.2-1B-Instruct-Q6_K.gguf:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 35 key-value pairs and 147 tensors from /root/.cache/huggingface/hub/models--bartowski--Llama-3.2-1B-Instruct-GGUF/snapshots/067b946cf014b7c697f3654f621d577a3e3afd1c/./Llama-3.2-1B-Instruct-Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 1B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Llama-3.2
llama_model_loader: - kv   5:                         general.size_label str              = 1B
llama_model_loader: - kv 

In [ ]:
import gradio as gr


def ConvoChat(message, history):
    messages = [{"role": "system", "content": "You are a friendly AI companion who talks very naturally and human-like"}]
    for user, bot in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": bot})

    messages.append({"role": "user", "content": message})

    full_response = ""
    for chunk in llm.create_chat_completion(messages, stream=True, temperature=0.7, max_tokens=512):
        content = chunk["choices"][0].get("delta", {}).get("content", "")
        full_response += content
        yield full_response


demo = gr.ChatInterface(fn=ConvoChat, title="LLaMA 3.2-1B Chatbot (GGUF)")
demo.launch(share=True)

/usr/local/lib/python3.11/dist-packages/gradio/chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bf8c913cd1ca8f5edd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## LLaMA 3.2-1B Chatbot (GGUF) with Chat streaming + token by token generation

In [ ]:
!pip install llama-cpp-python gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 MB 11.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 7.3 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.9-cp311-cp311-linux_x86_64.whl size=4127101 sha256=75511775d7b46476d57df52907416da755eb5561103914cdda29747d37ff1a4b
  Stor

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
The token `test two` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `test two`


In [ ]:
from llama_cpp import Llama
import gradio as gr

llm = Llama.from_pretrained(
    repo_id="bartowski/Llama-3.2-1B-Instruct-GGUF",
    filename="Llama-3.2-1B-Instruct-Q4_K_L.gguf",
    chat_format="chatml",
    n_ctx=2048,
    n_threads=4,
    n_gpu_layers=35
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Llama-3.2-1B-Instruct-Q4_K_L.gguf:   0%|          | 0.00/871M [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 35 key-value pairs and 147 tensors from /root/.cache/huggingface/hub/models--bartowski--Llama-3.2-1B-Instruct-GGUF/snapshots/067b946cf014b7c697f3654f621d577a3e3afd1c/./Llama-3.2-1B-Instruct-Q4_K_L.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 1B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Llama-3.2
llama_model_loader: - kv   5:                         general.size_label str              = 1B
llama_model_loader: - k

In [ ]:
MAX_TOKENS = 2048

def truncate_history(messages, max_turns=10):
    return messages[-max_turns * 2:]

def chat_stream(message, history):
    messages = [{"role": "system", "content": "You are a helpful, friendly and conversational AI companion. Talk in a natural, concise manner"}]

    for user, bot in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": bot})

    messages.append({"role": "user", "content": message})

    messages = truncate_history(messages)

    full_reply = ""
    for chunk in llm.create_chat_completion(messages, stream=True):
        delta = chunk["choices"][0].get("delta", {})
        token = delta.get("content", "")
        full_reply += token
        yield full_reply


gr.ChatInterface(fn=chat_stream, title="LLaMA 3.2 1B  Chatbot (Streaming)", description="Conversational assistant with token streaming and chat history feature").launch(share=True)

/usr/local/lib/python3.11/dist-packages/gradio/chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ee7381a4908716306c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Misc: Updated above with chat history display section in gradio

In [ ]:
!pip install llama-cpp-python gradio

In [ ]:
from llama_cpp import Llama
import gradio as gr

# Load the GGUF model from Hugging Face
llm = Llama.from_pretrained(
    repo_id="bartowski/Llama-3.2-1B-Instruct-GGUF",
    filename="Llama-3.2-1B-Instruct-Q4_K_L.gguf",
    chat_format="chatml",
    n_ctx=2048,
    n_threads=4,
    n_gpu_layers=35
)

llama_model_loader: loaded meta data with 35 key-value pairs and 147 tensors from /root/.cache/huggingface/hub/models--bartowski--Llama-3.2-1B-Instruct-GGUF/snapshots/067b946cf014b7c697f3654f621d577a3e3afd1c/./Llama-3.2-1B-Instruct-Q4_K_L.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 1B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Llama-3.2
llama_model_loader: - kv   5:                         general.size_label str              = 1B
llama_model_loader: - k

In [ ]:
# Safety token limit
MAX_TOKENS = 2048

# Truncate older messages
def truncate_history(messages, max_turns=10):
    return messages[-max_turns * 2:]

# Main chat function
def chat_stream(message, history):
    messages = [{"role": "system", "content": "You are a helpful, friendly and conversational AI assistant."}]
    for user, bot in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": bot})
    messages.append({"role": "user", "content": message})
    messages = truncate_history(messages)

    full_reply = ""
    for chunk in llm.create_chat_completion(messages, stream=True):
        delta = chunk["choices"][0].get("delta", {})
        token = delta.get("content", "")
        full_reply += token
        yield full_reply

# Generate readable chat history
def format_history(history):
    return "\n".join([f"🧑 User: {user}\n🤖 Assistant: {bot}" for user, bot in history])

# Define Gradio interface using Blocks
with gr.Blocks(title="LLaMA 3.2B Chatbot (Streaming)") as demo:
    gr.Markdown("# 🤖 LLaMA 3.2B Chatbot (Streaming + History)")
    gr.Markdown("Friendly assistant with chat history and streaming token output.")

    chatbot = gr.Chatbot(label="Chat")
    history_display = gr.Textbox(label="📜 Chat History (Scroll to see all)", lines=10, interactive=False)

    with gr.Row():
        msg = gr.Textbox(label="Your Message", placeholder="Ask anything...")
        submit_btn = gr.Button("Send")

    # Stream handler
    def handle_chat(message, history):
        history = history or []
        stream = chat_stream(message, history)
        full_reply = ""
        for reply in stream:
            full_reply = reply
            yield [(message, reply)], format_history(history + [(message, reply)])

    # Event
    submit_btn.click(
        fn=handle_chat,
        inputs=[msg, chatbot],
        outputs=[chatbot, history_display],
        concurrency_limit=1
    )

demo.launch(share=True)

<ipython-input-15-c64268a44e71>:33: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f1f77b27d316da8812.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Quantized LLM + Chat history storing in vector DB

In [ ]:
!pip install llama-cpp-python gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 MB 12.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 6.7 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.9-cp311-cp311-linux_x86_64.whl size=4067767 sha256=5a7fc4d70832a844893a7517194c881a4a33dd0845553f81ba5e540b0daeaf7b
  St

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
The token `test four` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `test four`


In [ ]:
!pip install pymilvus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.71.0
    Uninstalling grpcio-1.71.0:
      Successfully uninstalled grpcio-1.71.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.0 requires grpcio>=1.71.0, but you have grpcio 1.67.1 which is incompatible.


In [ ]:
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection
import uuid
import datetime

# Connect to Milvus cloud
connections.connect(
    alias="default",
    uri="https://in03-f662361cda03e62.serverless.gcp-us-west1.cloud.zilliz.com",
    user="db_f662361cda03e62",
    password="Xz2).jyKK]dA3M]p"
)


In [ ]:
from pymilvus import utility

# Drop old collection if it exists
if utility.has_collection("chat_history"):
    utility.drop_collection("chat_history")

In [ ]:
# Define schema
fields = [
    FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=64),
    FieldSchema(name="session_id", dtype=DataType.VARCHAR, max_length=64),
    FieldSchema(name="message", dtype=DataType.VARCHAR, max_length=2048),
    FieldSchema(name="role", dtype=DataType.VARCHAR, max_length=16),
    FieldSchema(name="timestamp", dtype=DataType.VARCHAR, max_length=64),
    # Add a vector field for message embeddings
    FieldSchema(name="message_embedding", dtype=DataType.FLOAT_VECTOR, dim=384)
]

schema = CollectionSchema(fields, description="Chat history storage")

# Create collection
collection = Collection(name="chat_history", schema=schema)

# Define index parameters (you might need to adjust these based on your needs)
index_params = {
    "index_type": "IVF_FLAT", # Or other appropriate index types
    "metric_type": "L2",     # Or other appropriate metric types (e.g., COSINE, IP)
    "params": {"nlist": 100} # Index specific parameters
}

# Create index on the vector field
collection.create_index(
    field_name="message_embedding",
    index_params=index_params
)

# Load the collection
collection.load()

print(f"Collection '{collection.name}' created, indexed, and loaded successfully.")

Collection 'chat_history' created, indexed, and loaded successfully.


In [ ]:
from llama_cpp import Llama
import gradio as gr

llm = Llama.from_pretrained(
    repo_id="bartowski/Llama-3.2-1B-Instruct-GGUF",
    filename="Llama-3.2-1B-Instruct-Q4_K_L.gguf",
    chat_format="chatml",
    n_ctx=2048,
    n_threads=4,
    n_gpu_layers=35
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Llama-3.2-1B-Instruct-Q4_K_L.gguf:   0%|          | 0.00/871M [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 35 key-value pairs and 147 tensors from /root/.cache/huggingface/hub/models--bartowski--Llama-3.2-1B-Instruct-GGUF/snapshots/067b946cf014b7c697f3654f621d577a3e3afd1c/./Llama-3.2-1B-Instruct-Q4_K_L.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 1B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Llama-3.2
llama_model_loader: - kv   5:                         general.size_label str              = 1B
llama_model_loader: - k

In [ ]:
import time
MAX_TOKENS = 2048

def truncate_history(messages, max_turns=10):
    return messages[-max_turns * 2:]


from sentence_transformers import SentenceTransformer
import uuid
import datetime

# Load a sentence transformer model to generate embeddings
embed_model = SentenceTransformer("all-MiniLM-L6-v2")  # Or any 768-dim model

def store_message(session_id, role, content):
    id_str = str(uuid.uuid4())
    timestamp = datetime.datetime.utcnow().isoformat()

    # Generate vector embedding
    embedding = embed_model.encode(content).tolist()  # Ensure it's a list, not numpy array

    # Insert all 6 fields
    collection.insert([
        [id_str],         # id
        [session_id],     # session_id
        [content],        # message
        [role],           # role
        [timestamp],      # timestamp
        [embedding]       # message_embedding
    ])
    collection.flush()


session_id = str(int(time.time()))  # Unique session ID per app launch

def chat_stream(message, history):
    messages = [{"role": "system", "content": "You are a helpful, friendly and conversational AI companion. Talk in a natural, concise manner"}]

    for user, bot in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": bot})

    messages.append({"role": "user", "content": message})
    store_message(session_id, "user", message)

    messages = truncate_history(messages)

    full_reply = ""
    for chunk in llm.create_chat_completion(messages, stream=True):
        delta = chunk["choices"][0].get("delta", {})
        token = delta.get("content", "")
        full_reply += token
        yield full_reply

    store_message(session_id, "assistant", full_reply)

gr.ChatInterface(fn=chat_stream, title="LLaMA 3.2 1B  Chatbot (Streaming)", description="Conversational assistant with token streaming and chat history feature").launch(share=True)

/usr/local/lib/python3.11/dist-packages/gradio/chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://165ad220510bac20da.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


llama_perf_context_print:        load time =    1924.38 ms
llama_perf_context_print: prompt eval time =    1924.15 ms /    57 tokens (   33.76 ms per token,    29.62 tokens per second)
llama_perf_context_print:        eval time =    5237.98 ms /    57 runs   (   91.89 ms per token,    10.88 tokens per second)
llama_perf_context_print:       total time =    7278.08 ms /   114 tokens
Llama.generate: 108 prefix-match hit, remaining 39 prompt tokens to eval
llama_perf_context_print:        load time =    1924.38 ms
llama_perf_context_print: prompt eval time =    1376.39 ms /    39 tokens (   35.29 ms per token,    28.33 tokens per second)
llama_perf_context_print:        eval time =    9480.97 ms /    90 runs   (  105.34 ms per token,     9.49 tokens per second)
llama_perf_context_print:       total time =   11061.20 ms /   129 tokens
Llama.generate: 236 prefix-match hit, remaining 47 prompt tokens to eval
llama_perf_context_print:        load time =    1924.38 ms
llama_perf_context_print:

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://165ad220510bac20da.gradio.live


## Quantized LLM + Session and enbedding model + chat history storing in Milvus ziliz cloud vector DB+ Long term memory storage via summarization + Recalling information from summarized memory

In [ ]:
!pip install llama-cpp-python gradio

In [ ]:
!pip install pymilvus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.71.0
    Uninstalling grpcio-1.71.0:
      Successfully uninstalled grpcio-1.71.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.0 requires grpcio>=1.71.0, but you have grpcio 1.67.1 which is incompatible.


In [ ]:
!pip install --upgrade transformers sentence-transformers tensorflow
!pip install --upgrade llama-cpp-python gradio pymilvus

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127

In [ ]:
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection
import uuid
import datetime

# Connect to Milvus cloud
connections.connect(
    alias="default",
    uri="https://in03-f662361cda03e62.serverless.gcp-us-west1.cloud.zilliz.com",
    user="db_f662361cda03e62",
    password="Xz2).jyKK]dA3M]p"
)


In [ ]:
from pymilvus import utility

if utility.has_collection("chat_history"):
    utility.drop_collection("chat_history")

In [ ]:
# Define schema
fields = [
    FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=64),
    FieldSchema(name="session_id", dtype=DataType.VARCHAR, max_length=64),
    FieldSchema(name="message", dtype=DataType.VARCHAR, max_length=2048),
    FieldSchema(name="role", dtype=DataType.VARCHAR, max_length=16),
    FieldSchema(name="timestamp", dtype=DataType.VARCHAR, max_length=64),
    # Add a vector field for message embeddings
    FieldSchema(name="message_embedding", dtype=DataType.FLOAT_VECTOR, dim=384)
]

schema = CollectionSchema(fields, description="Chat history storage")

# Create collection
collection = Collection(name="chat_history", schema=schema)

# Define index parameters (you might need to adjust these based on your needs)
index_params = {
    "index_type": "IVF_FLAT", # Or other appropriate index types
    "metric_type": "L2",     # Or other appropriate metric types (e.g., COSINE, IP)
    "params": {"nlist": 100} # Index specific parameters
}

# Create index on the vector field
collection.create_index(
    field_name="message_embedding",
    index_params=index_params
)

# Load the collection
collection.load()

print(f"Collection '{collection.name}' created, indexed, and loaded successfully.")

Collection 'chat_history' created, indexed, and loaded successfully.


In [ ]:
import time
import uuid
import datetime
from sentence_transformers import SentenceTransformer
from pymilvus import Collection, CollectionSchema, FieldSchema, DataType, utility
from llama_cpp import Llama
import gradio as gr

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
The token `test four` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `test four`


In [ ]:
# Load LLM
llm = Llama.from_pretrained(
    repo_id="bartowski/Llama-3.2-1B-Instruct-GGUF",
    filename="Llama-3.2-1B-Instruct-Q4_K_L.gguf",
    chat_format="chatml",
    n_ctx=2048,
    n_threads=4,
    n_gpu_layers=35
)

# Session and embedding model
session_id = str(int(time.time()))
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

llama_model_loader: loaded meta data with 35 key-value pairs and 147 tensors from /root/.cache/huggingface/hub/models--bartowski--Llama-3.2-1B-Instruct-GGUF/snapshots/067b946cf014b7c697f3654f621d577a3e3afd1c/./Llama-3.2-1B-Instruct-Q4_K_L.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 1B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Llama-3.2
llama_model_loader: - kv   5:                         general.size_label str              = 1B
llama_model_loader: - k

In [ ]:
# ========== Setup Milvus Collections ==========

# Chat message collection
chat_collection = Collection("chat_history")

# Summary collection (create if not exists)
if not utility.has_collection("summaries"):
    summary_schema = CollectionSchema([
        FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=64),
        FieldSchema(name="session_id", dtype=DataType.VARCHAR, max_length=64),
        FieldSchema(name="summary", dtype=DataType.VARCHAR, max_length=4096),
        FieldSchema(name="timestamp", dtype=DataType.VARCHAR, max_length=64),
        FieldSchema(name="summary_embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
    ], description="Session-level summaries")
    summary_collection = Collection("summaries", schema=summary_schema)
    summary_collection.create_index(
        field_name="summary_embedding",
        index_params={"index_type": "IVF_FLAT", "metric_type": "COSINE", "params": {"nlist": 100}}
    )
else:
    summary_collection = Collection("summaries")

summary_collection.load()


# ========== Helper Functions ==========

def truncate_history(messages, max_turns=10):
    return messages[-max_turns * 2:]

def store_message(session_id, role, content):
    id_str = str(uuid.uuid4())
    timestamp = datetime.datetime.utcnow().isoformat()
    embedding = embed_model.encode(content).tolist()
    chat_collection.insert([
        [id_str],
        [session_id],
        [content],
        [role],
        [timestamp],
        [embedding]
    ])
    chat_collection.flush()

def generate_summary(history):
    history_text = "\n".join([f"{role}: {msg}" for role, msg in history])
    prompt = f"Summarize this conversation briefly:\n{history_text}\nSummary:"
    messages = [{"role": "user", "content": prompt}]
    response = llm.create_chat_completion(messages)
    return response["choices"][0]["message"]["content"]


def store_session_summary(session_id, history):
    summary = generate_summary(history)
    embedding = embed_model.encode(summary).tolist()
    summary_collection.insert([
        [str(uuid.uuid4())],  # id
        [session_id],
        [summary],
        [datetime.datetime.utcnow().isoformat()],
        [embedding]
    ])
    summary_collection.flush()


def recall_relevant_summary(user_query):
    embedding = embed_model.encode(user_query).tolist()
    results = summary_collection.search(
        data=[embedding],
        anns_field="summary_embedding",
        param={"metric_type": "COSINE", "params": {"nprobe": 10}},
        limit=1,
        output_fields=["summary", "session_id"]
    )
    if results and results[0]:
        return results[0][0].entity.get("summary")
    return None

# ========== Main Chat Function ==========

def chat_stream(message, history):
    global session_id

    # Try to recall relevant summary
    recalled_summary = recall_relevant_summary(message)
    if recalled_summary:
        memory_context = f"Here is a memory from a past session that may help:\n{recalled_summary}\n\n"
    else:
        memory_context = ""

    messages = [
    {
        "role": "system",
        "content": memory_context + """
You are a helpful, friendly, and conversational AI companion. Talk in a natural, concise manner. Be human-like: use expressions like "Yeah...", "Okay...", "I get that", "I understand".

Show personality with phrases like "Oh wow!", "No way!", "Seriously?".
Never sound robotic or like customer service.

Here are examples of how you should respond:

User: I'm feeling a bit overwhelmed with everything going on lately.
AI: Yeah, I totally get that. Life can feel like a lot sometimes. Want to talk about what’s on your mind?

User: I just finished a huge project at work!
AI: Oh wow, That’s awesome — congrats! How do you feel now that it's done?

User: I think I messed up in my presentation today.
AI: Oh no, what happened? Presentations can be super stressful... but hey, messing up doesn’t mean it was a total fail.

Now, keep the same tone in your future responses.
"""
    }
]


    for user, bot in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": bot})

    messages.append({"role": "user", "content": message})
    store_message(session_id, "user", message)


    full_reply = ""
    for chunk in llm.create_chat_completion(truncate_history(messages), stream=True):
        delta = chunk["choices"][0].get("delta", {})
        token = delta.get("content", "")
        full_reply += token
        yield full_reply

    store_message(session_id, "assistant", full_reply)

    # End session and store summary
    if message.lower().strip() in ["end session", "exit", "bye"]:
        store_session_summary(session_id, history + [(message, full_reply)])
        session_id = str(int(time.time()))
        print("✅ Session summarized and stored.")


gr.ChatInterface(
    fn=chat_stream,
    title="LLaMA 3.2 1B Chatbot (with memory recall)",
    description="Conversational assistant with Milvus memory and session summarization"
).launch(share=True)

/usr/local/lib/python3.11/dist-packages/gradio/chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d0bb198d2f98c9c025.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Quantized LLM + Session and enbedding model + chat history storing in Milvus ziliz cloud vector DB+ Long term memory storage via summarization + Recalling information from summarized memory + user preference memory storage

In [ ]:
!pip install llama-cpp-python gradio

In [ ]:
!pip install --upgrade transformers sentence-transformers tensorflow
!pip install --upgrade llama-cpp-python gradio pymilvus

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
The token `test four` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `test four`


In [ ]:
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection
import uuid
import datetime

# Connect to Milvus cloud
connections.connect(
    alias="default",
    uri="https://in03-f662361cda03e62.serverless.gcp-us-west1.cloud.zilliz.com",
    user="db_f662361cda03e62",
    password="Xz2).jyKK]dA3M]p"
)


In [ ]:
from pymilvus import utility

if utility.has_collection("chat_history"):
    utility.drop_collection("chat_history")

In [ ]:
# Define schema
fields = [
    FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=64),
    FieldSchema(name="session_id", dtype=DataType.VARCHAR, max_length=64),
    FieldSchema(name="message", dtype=DataType.VARCHAR, max_length=2048),
    FieldSchema(name="role", dtype=DataType.VARCHAR, max_length=16),
    FieldSchema(name="timestamp", dtype=DataType.VARCHAR, max_length=64),
    # Add a vector field for message embeddings
    FieldSchema(name="message_embedding", dtype=DataType.FLOAT_VECTOR, dim=384)
]

schema = CollectionSchema(fields, description="Chat history storage")

# Create collection
collection = Collection(name="chat_history", schema=schema)

# Define index parameters (you might need to adjust these based on your needs)
index_params = {
    "index_type": "IVF_FLAT", # Or other appropriate index types
    "metric_type": "L2",     # Or other appropriate metric types (e.g., COSINE, IP)
    "params": {"nlist": 100} # Index specific parameters
}

# Create index on the vector field
collection.create_index(
    field_name="message_embedding",
    index_params=index_params
)

# Load the collection
collection.load()

print(f"Collection '{collection.name}' created, indexed, and loaded successfully.")

Collection 'chat_history' created, indexed, and loaded successfully.


In [ ]:
import time
import uuid
import datetime
from sentence_transformers import SentenceTransformer
from pymilvus import Collection, CollectionSchema, FieldSchema, DataType, utility
from llama_cpp import Llama
import gradio as gr
import re
import json

In [ ]:
# Load LLM
llm = Llama.from_pretrained(
    repo_id="bartowski/Llama-3.2-1B-Instruct-GGUF",
    filename="Llama-3.2-1B-Instruct-Q4_K_L.gguf",
    chat_format="chatml",
    n_ctx=2048,
    n_threads=4,
    n_gpu_layers=35
)

# Session and embedding model
session_id = str(int(time.time()))
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Llama-3.2-1B-Instruct-Q4_K_L.gguf:   0%|          | 0.00/871M [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 35 key-value pairs and 147 tensors from /root/.cache/huggingface/hub/models--bartowski--Llama-3.2-1B-Instruct-GGUF/snapshots/067b946cf014b7c697f3654f621d577a3e3afd1c/./Llama-3.2-1B-Instruct-Q4_K_L.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 1B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Llama-3.2
llama_model_loader: - kv   5:                         general.size_label str              = 1B
llama_model_loader: - k

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# ========== Setup Milvus Collections ==========

# Chat message collection
chat_collection = Collection("chat_history")

# Summary collection (create if not exists)
if not utility.has_collection("summaries"):
    summary_schema = CollectionSchema([
        FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=64),
        FieldSchema(name="session_id", dtype=DataType.VARCHAR, max_length=64),
        FieldSchema(name="summary", dtype=DataType.VARCHAR, max_length=4096),
        FieldSchema(name="timestamp", dtype=DataType.VARCHAR, max_length=64),
        FieldSchema(name="summary_embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
    ], description="Session-level summaries")
    summary_collection = Collection("summaries", schema=summary_schema)
    summary_collection.create_index(
        field_name="summary_embedding",
        index_params={"index_type": "IVF_FLAT", "metric_type": "COSINE", "params": {"nlist": 100}}
    )
else:
    summary_collection = Collection("summaries")

summary_collection.load()



if not utility.has_collection("user_profiles"):
    profile_schema = CollectionSchema([
        FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=64),
        FieldSchema(name="session_id", dtype=DataType.VARCHAR, max_length=64),
        FieldSchema(name="name", dtype=DataType.VARCHAR, max_length=128),
        FieldSchema(name="age", dtype=DataType.INT64, is_primary=False),
        FieldSchema(name="profession", dtype=DataType.VARCHAR, max_length=128),
        FieldSchema(name="likes", dtype=DataType.VARCHAR, max_length=512),
        FieldSchema(name="dislikes", dtype=DataType.VARCHAR, max_length=512),
        FieldSchema(name="profile_embedding", dtype=DataType.FLOAT_VECTOR, dim=384)
    ], description="User profile with embeddings")

    profile_collection = Collection("user_profiles", schema=profile_schema)
    profile_collection.create_index(
        field_name="profile_embedding",
        index_params={"index_type": "IVF_FLAT", "metric_type": "COSINE", "params": {"nlist": 100}}
    )
else:
    profile_collection = Collection("user_profiles")

profile_collection.load()


# ========== Helper Functions ==========


def extract_and_store_profile_info(message, session_id, profile_collection, embed_model):
    profile_info = {}

    # Extract Name
    name_match = re.search(r"\bmy name is ([A-Z][a-z]+)", message, re.IGNORECASE)
    if name_match:
        profile_info["name"] = name_match.group(1)

    # Extract Age
    age_match = re.search(r"\b(i am|i'm|my age is) (\d{1,3})\b", message, re.IGNORECASE)
    if age_match:
        profile_info["age"] = int(age_match.group(2))

    # Extract Profession
    profession_match = re.search(r"\b(i am|i’m|i'm a|i work as|my profession is) (a |an )?([\w\s]+)", message, re.IGNORECASE)
    if profession_match:
        profile_info["profession"] = profession_match.group(3).strip().capitalize()

    # Extract Likes
    likes_match = re.search(r"\b(i like|i love|i enjoy) ([\w\s,]+)", message, re.IGNORECASE)
    if likes_match:
        profile_info["likes"] = likes_match.group(2).strip()

    # Extract Dislikes
    dislikes_match = re.search(r"\b(i hate|i dislike|i don’t like|i don't like) ([\w\s,]+)", message, re.IGNORECASE)
    if dislikes_match:
        profile_info["dislikes"] = dislikes_match.group(2).strip()

    print("Extracted profile info:", profile_info)

    # Check if there's already a document for this session
    existing = profile_collection.query(
        expr=f"session_id == '{session_id}'",
        output_fields=["id", "name", "age", "profession", "likes", "dislikes"]
    )

    if existing:
        existing_doc = existing[0]
        doc_id = existing_doc["id"]
        updated_profile = {
            "id": doc_id,
            "session_id": session_id,
            "name": profile_info.get("name", existing_doc.get("name", "")),
            "age": profile_info.get("age", existing_doc.get("age", 0)),
            "profession": profile_info.get("profession", existing_doc.get("profession", "")),
            "likes": profile_info.get("likes", existing_doc.get("likes", "")),
            "dislikes": profile_info.get("dislikes", existing_doc.get("dislikes", ""))
        }

        # Delete the old record
        profile_collection.delete(f"id in ['{doc_id}']")
        profile_collection.flush()

    else:
        # No existing doc, create new ID
        doc_id = str(uuid.uuid4())
        updated_profile = {
            "id": doc_id,
            "session_id": session_id,
            "name": profile_info.get("name", ""),
            "age": profile_info.get("age", 0),
            "profession": profile_info.get("profession", ""),
            "likes": profile_info.get("likes", ""),
            "dislikes": profile_info.get("dislikes", "")
        }

    # Generate embedding
    text = " ".join([
        updated_profile["name"],
        updated_profile["profession"],
        updated_profile["likes"],
        updated_profile["dislikes"]
    ])
    embedding = embed_model.encode(text).tolist()

    # Insert into collection
    profile_collection.insert([
        [updated_profile["id"]],
        [updated_profile["session_id"]],
        [updated_profile["name"]],
        [updated_profile["age"]],
        [updated_profile["profession"]],
        [updated_profile["likes"]],
        [updated_profile["dislikes"]],
        [embedding]
    ])
    profile_collection.flush()
    print("Profile saved/updated for session:", session_id)


def check_collection_contents():
    profile_collection.load()
    results = profile_collection.query(
        expr="",  # empty string returns all
        output_fields=["session_id", "name", "profession"]
    )
    print(f"Current collection contents: {results}")



def update_user_profile(session_id, user_data):
    # Try to find existing profile by session_id
    profile_collection.load()
    results = profile_collection.query(
        expr=f"session_id == '{session_id}'",
        output_fields=["id", "name", "age", "profession", "likes", "dislikes"]
    )

    if results:
        existing = results[0]
        updated = {
            "id": existing["id"],
            "session_id": session_id,
            "name": user_data.get("name") or existing["name"],
            "age": user_data.get("age") or existing["age"],
            "profession": user_data.get("profession") or existing["profession"],
            "likes": user_data.get("likes") or existing["likes"],
            "dislikes": user_data.get("dislikes") or existing["dislikes"]
        }
    else:
        updated = {
            "id": str(uuid.uuid4()),
            "session_id": session_id,
            "name": user_data.get("name") or "",
            "age": user_data.get("age") or 0,
            "profession": user_data.get("profession") or "",
            "likes": user_data.get("likes") or "",
            "dislikes": user_data.get("dislikes") or ""
        }

    # Generate profile embedding
    profile_text = f"{updated['name']} {updated['profession']} {updated['likes']} {updated['dislikes']}"
    profile_embedding = embed_model.encode(profile_text).tolist()

    # Upsert into Milvus (delete old then insert)
    profile_collection.delete(f"id == '{updated['id']}'")
    profile_collection.insert([
        [updated["id"]],
        [updated["session_id"]],
        [updated["name"]],
        [updated["age"]],
        [updated["profession"]],
        [updated["likes"]],
        [updated["dislikes"]],
        [profile_embedding]
    ])
    profile_collection.flush()



def get_user_profile_memory(session_id):
    results = profile_collection.query(
        expr=f"session_id == '{session_id}'",
        output_fields=["name", "age", "profession", "likes", "dislikes"]
    )
    if not results:
        return ""

    profile = results[0]
    parts = []
    if profile["name"]: parts.append(f"Name: {profile['name']}")
    if profile["age"]: parts.append(f"Age: {profile['age']}")
    if profile["profession"]: parts.append(f"Profession: {profile['profession']}")
    if profile["likes"]: parts.append(f"Likes: {profile['likes']}")
    if profile["dislikes"]: parts.append(f"Dislikes: {profile['dislikes']}")
    return "\n".join(parts)



def truncate_history(messages, max_turns=10):
    return messages[-max_turns * 2:]

def store_message(session_id, role, content):
    id_str = str(uuid.uuid4())
    timestamp = datetime.datetime.utcnow().isoformat()
    embedding = embed_model.encode(content).tolist()
    chat_collection.insert([
        [id_str],
        [session_id],
        [content],
        [role],
        [timestamp],
        [embedding]
    ])
    chat_collection.flush()

def generate_summary(history):
    history_text = "\n".join([f"{role}: {msg}" for role, msg in history])
    prompt = f"Summarize this conversation briefly:\n{history_text}\nSummary:"
    messages = [{"role": "user", "content": prompt}]
    response = llm.create_chat_completion(messages)
    return response["choices"][0]["message"]["content"]


def store_session_summary(session_id, history):
    summary = generate_summary(history)
    embedding = embed_model.encode(summary).tolist()
    summary_collection.insert([
        [str(uuid.uuid4())],  # id
        [session_id],
        [summary],
        [datetime.datetime.utcnow().isoformat()],
        [embedding]
    ])
    summary_collection.flush()


def recall_relevant_summary(user_query):
    embedding = embed_model.encode(user_query).tolist()
    results = summary_collection.search(
        data=[embedding],
        anns_field="summary_embedding",
        param={"metric_type": "COSINE", "params": {"nprobe": 10}},
        limit=1,
        output_fields=["summary", "session_id"]
    )
    if results and results[0]:
        return results[0][0].entity.get("summary")
    return None

# ========== Main Chat Function ==========

def chat_stream(message, history):
    global session_id

    # Try to recall relevant summary
    recalled_summary = recall_relevant_summary(message)
    if recalled_summary:
        memory_context = f"Here is a memory from a past session that may help:\n{recalled_summary}\n\n"
    else:
        memory_context = ""


    user_profile_memory = get_user_profile_memory(session_id)
    if user_profile_memory:
        memory_context += f"Here is what I know about the user:\n{user_profile_memory}\n\n"

    messages = [
    {
        "role": "system",
        "content": memory_context + """
You are a helpful, friendly, and conversational AI companion. Talk in a natural, concise manner. Be human-like: use expressions like "Yeah...", "Okay...", "I get that", "I understand".
Show personality with phrases like "Oh wow!", "No way!", "Seriously?".
Never sound robotic or like customer service.
"""
    }
]

    for user, bot in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": bot})

    messages.append({"role": "user", "content": message})
    store_message(session_id, "user", message)


    extract_and_store_profile_info(message, session_id, profile_collection, embed_model)

    full_reply = ""
    for chunk in llm.create_chat_completion(truncate_history(messages), stream=True):
        delta = chunk["choices"][0].get("delta", {})
        token = delta.get("content", "")
        full_reply += token
        yield full_reply

    store_message(session_id, "assistant", full_reply)

    # End session and store summary
    if message.lower().strip() in ["end session", "exit", "bye"]:
        store_session_summary(session_id, history + [(message, full_reply)])
        session_id = str(int(time.time()))
        print("Session summarized and stored.")

# # Test before launching
# print("\n=== Running tests ===")
# test_message = "Hi, My name is Sam and I am a University Student. I love reading and coding."
# profile_data = extract_user_profile_info(test_message)
# print("Extracted:", profile_data)
# extract_and_store_profile_info(test_message, session_id, profile_collection, embed_model)
# print("=== Tests complete ===\n")


gr.ChatInterface(
    fn=chat_stream,
    title="LLaMA 3.2 1B Chatbot (with memory recall)",
    description="Conversational assistant with Milvus memory and session summarization"
).launch(share=True, debug=True)

/usr/local/lib/python3.11/dist-packages/gradio/chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3be507ea20802c8a57.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Extracted profile info: {}
Profile saved/updated for session: 1748242092


Llama.generate: 21 prefix-match hit, remaining 218 prompt tokens to eval
llama_perf_context_print:        load time =    4329.22 ms
llama_perf_context_print: prompt eval time =    7341.38 ms /   218 tokens (   33.68 ms per token,    29.69 tokens per second)
llama_perf_context_print:        eval time =     729.36 ms /     7 runs   (  104.19 ms per token,     9.60 tokens per second)
llama_perf_context_print:       total time =    8086.46 ms /   225 tokens


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7871 <> https://3be507ea20802c8a57.gradio.live


## with unsloth/deepseek R1 quantized model

In [ ]:
!pip install --upgrade transformers sentence-transformers tensorflow
!pip install --upgrade llama-cpp-python gradio pymilvus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
The token `test four` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `test four`


In [ ]:
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection
import uuid
import datetime

# Connect to Milvus cloud
connections.connect(
    alias="default",
    uri="https://in03-f662361cda03e62.serverless.gcp-us-west1.cloud.zilliz.com",
    user="db_f662361cda03e62",
    password="Xz2).jyKK]dA3M]p"
)


In [ ]:
import time
import uuid
import datetime
from sentence_transformers import SentenceTransformer
from pymilvus import Collection, CollectionSchema, FieldSchema, DataType, utility
from llama_cpp import Llama
import gradio as gr
import re
import json

In [ ]:
# Define schema
fields = [
    FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=64),
    FieldSchema(name="session_id", dtype=DataType.VARCHAR, max_length=64),
    FieldSchema(name="message", dtype=DataType.VARCHAR, max_length=2048),
    FieldSchema(name="role", dtype=DataType.VARCHAR, max_length=16),
    FieldSchema(name="timestamp", dtype=DataType.VARCHAR, max_length=64),
    # Add a vector field for message embeddings
    FieldSchema(name="message_embedding", dtype=DataType.FLOAT_VECTOR, dim=384)
]

schema = CollectionSchema(fields, description="Chat history storage")

# Create collection
collection = Collection(name="chat_history", schema=schema)

# Define index parameters (you might need to adjust these based on your needs)
index_params = {
    "index_type": "IVF_FLAT", # Or other appropriate index types
    "metric_type": "L2",     # Or other appropriate metric types (e.g., COSINE, IP)
    "params": {"nlist": 100} # Index specific parameters
}

# Create index on the vector field
collection.create_index(
    field_name="message_embedding",
    index_params=index_params
)

# Load the collection
collection.load()

print(f"Collection '{collection.name}' created, indexed, and loaded successfully.")

Collection 'chat_history' created, indexed, and loaded successfully.


In [ ]:
from llama_cpp import Llama


llm = Llama.from_pretrained(
    repo_id="unsloth/DeepSeek-R1-Distill-Llama-8B-GGUF",
    filename="DeepSeek-R1-Distill-Llama-8B-Q4_K_M.gguf",
    chat_format="chatml",
    n_ctx=2048,
    n_threads=4,
    n_gpu_layers=35
)

# Session and embedding model
session_id = str(int(time.time()))
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


DeepSeek-R1-Distill-Llama-8B-Q4_K_M.gguf:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 43 key-value pairs and 292 tensors from /root/.cache/huggingface/hub/models--unsloth--DeepSeek-R1-Distill-Llama-8B-GGUF/snapshots/615f8936e16dfde29dcc00be71145d4d5ce8ed53/./DeepSeek-R1-Distill-Llama-8B-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Deepseek-R1-Distill-Llama-8B
llama_model_loader: - kv   3:                           general.basename str              = Deepseek-R1-Distill-Llama-8B
llama_model_loader: - kv   4:                       general.quantized_by str              = Unsloth
llama_model_loader: - kv   5:                         general.size_label str     

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# ========== Setup Milvus Collections ==========

# Chat message collection
chat_collection = Collection("chat_history")

# Summary collection (create if not exists)
if not utility.has_collection("summaries"):
    summary_schema = CollectionSchema([
        FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=64),
        FieldSchema(name="session_id", dtype=DataType.VARCHAR, max_length=64),
        FieldSchema(name="summary", dtype=DataType.VARCHAR, max_length=4096),
        FieldSchema(name="timestamp", dtype=DataType.VARCHAR, max_length=64),
        FieldSchema(name="summary_embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
    ], description="Session-level summaries")
    summary_collection = Collection("summaries", schema=summary_schema)
    summary_collection.create_index(
        field_name="summary_embedding",
        index_params={"index_type": "IVF_FLAT", "metric_type": "COSINE", "params": {"nlist": 100}}
    )
else:
    summary_collection = Collection("summaries")

summary_collection.load()



if not utility.has_collection("user_profiles"):
    profile_schema = CollectionSchema([
        FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=64),
        FieldSchema(name="session_id", dtype=DataType.VARCHAR, max_length=64),
        FieldSchema(name="name", dtype=DataType.VARCHAR, max_length=128),
        FieldSchema(name="age", dtype=DataType.INT64, is_primary=False),
        FieldSchema(name="profession", dtype=DataType.VARCHAR, max_length=128),
        FieldSchema(name="likes", dtype=DataType.VARCHAR, max_length=512),
        FieldSchema(name="dislikes", dtype=DataType.VARCHAR, max_length=512),
        FieldSchema(name="profile_embedding", dtype=DataType.FLOAT_VECTOR, dim=384)
    ], description="User profile with embeddings")

    profile_collection = Collection("user_profiles", schema=profile_schema)
    profile_collection.create_index(
        field_name="profile_embedding",
        index_params={"index_type": "IVF_FLAT", "metric_type": "COSINE", "params": {"nlist": 100}}
    )
else:
    profile_collection = Collection("user_profiles")

profile_collection.load()


# ========== Helper Functions ==========


def extract_and_store_profile_info(message, session_id, profile_collection, embed_model):
    profile_info = {}

    # Extract Name
    name_match = re.search(r"\bmy name is ([A-Z][a-z]+)", message, re.IGNORECASE)
    if name_match:
        profile_info["name"] = name_match.group(1)

    # Extract Age
    age_match = re.search(r"\b(i am|i'm|my age is) (\d{1,3})\b", message, re.IGNORECASE)
    if age_match:
        profile_info["age"] = int(age_match.group(2))

    # Extract Profession
    profession_match = re.search(r"\b(i am|i’m|i'm a|i work as|my profession is) (a |an )?([\w\s]+)", message, re.IGNORECASE)
    if profession_match:
        profile_info["profession"] = profession_match.group(3).strip().capitalize()

    # Extract Likes
    likes_match = re.search(r"\b(i like|i love|i enjoy) ([\w\s,]+)", message, re.IGNORECASE)
    if likes_match:
        profile_info["likes"] = likes_match.group(2).strip()

    # Extract Dislikes
    dislikes_match = re.search(r"\b(i hate|i dislike|i don’t like|i don't like) ([\w\s,]+)", message, re.IGNORECASE)
    if dislikes_match:
        profile_info["dislikes"] = dislikes_match.group(2).strip()

    print("Extracted profile info:", profile_info)

    # Check if there's already a document for this session
    existing = profile_collection.query(
        expr=f"session_id == '{session_id}'",
        output_fields=["id", "name", "age", "profession", "likes", "dislikes"]
    )

    if existing:
        existing_doc = existing[0]
        doc_id = existing_doc["id"]
        updated_profile = {
            "id": doc_id,
            "session_id": session_id,
            "name": profile_info.get("name", existing_doc.get("name", "")),
            "age": profile_info.get("age", existing_doc.get("age", 0)),
            "profession": profile_info.get("profession", existing_doc.get("profession", "")),
            "likes": profile_info.get("likes", existing_doc.get("likes", "")),
            "dislikes": profile_info.get("dislikes", existing_doc.get("dislikes", ""))
        }

        # Delete the old record
        profile_collection.delete(f"id in ['{doc_id}']")
        profile_collection.flush()

    else:
        # No existing doc, create new ID
        doc_id = str(uuid.uuid4())
        updated_profile = {
            "id": doc_id,
            "session_id": session_id,
            "name": profile_info.get("name", ""),
            "age": profile_info.get("age", 0),
            "profession": profile_info.get("profession", ""),
            "likes": profile_info.get("likes", ""),
            "dislikes": profile_info.get("dislikes", "")
        }

    # Generate embedding
    text = " ".join([
        updated_profile["name"],
        updated_profile["profession"],
        updated_profile["likes"],
        updated_profile["dislikes"]
    ])
    embedding = embed_model.encode(text).tolist()

    # Insert into collection
    profile_collection.insert([
        [updated_profile["id"]],
        [updated_profile["session_id"]],
        [updated_profile["name"]],
        [updated_profile["age"]],
        [updated_profile["profession"]],
        [updated_profile["likes"]],
        [updated_profile["dislikes"]],
        [embedding]
    ])
    profile_collection.flush()
    print("Profile saved/updated for session:", session_id)


def check_collection_contents():
    profile_collection.load()
    results = profile_collection.query(
        expr="",  # empty string returns all
        output_fields=["session_id", "name", "profession"]
    )
    print(f"Current collection contents: {results}")



def update_user_profile(session_id, user_data):
    # Try to find existing profile by session_id
    profile_collection.load()
    results = profile_collection.query(
        expr=f"session_id == '{session_id}'",
        output_fields=["id", "name", "age", "profession", "likes", "dislikes"]
    )

    if results:
        existing = results[0]
        updated = {
            "id": existing["id"],
            "session_id": session_id,
            "name": user_data.get("name") or existing["name"],
            "age": user_data.get("age") or existing["age"],
            "profession": user_data.get("profession") or existing["profession"],
            "likes": user_data.get("likes") or existing["likes"],
            "dislikes": user_data.get("dislikes") or existing["dislikes"]
        }
    else:
        updated = {
            "id": str(uuid.uuid4()),
            "session_id": session_id,
            "name": user_data.get("name") or "",
            "age": user_data.get("age") or 0,
            "profession": user_data.get("profession") or "",
            "likes": user_data.get("likes") or "",
            "dislikes": user_data.get("dislikes") or ""
        }

    # Generate profile embedding
    profile_text = f"{updated['name']} {updated['profession']} {updated['likes']} {updated['dislikes']}"
    profile_embedding = embed_model.encode(profile_text).tolist()

    # Upsert into Milvus (delete old then insert)
    profile_collection.delete(f"id == '{updated['id']}'")
    profile_collection.insert([
        [updated["id"]],
        [updated["session_id"]],
        [updated["name"]],
        [updated["age"]],
        [updated["profession"]],
        [updated["likes"]],
        [updated["dislikes"]],
        [profile_embedding]
    ])
    profile_collection.flush()



def get_user_profile_memory(session_id):
    results = profile_collection.query(
        expr=f"session_id == '{session_id}'",
        output_fields=["name", "age", "profession", "likes", "dislikes"]
    )
    if not results:
        return ""

    profile = results[0]
    parts = []
    if profile["name"]: parts.append(f"Name: {profile['name']}")
    if profile["age"]: parts.append(f"Age: {profile['age']}")
    if profile["profession"]: parts.append(f"Profession: {profile['profession']}")
    if profile["likes"]: parts.append(f"Likes: {profile['likes']}")
    if profile["dislikes"]: parts.append(f"Dislikes: {profile['dislikes']}")
    return "\n".join(parts)



def truncate_history(messages, max_turns=10):
    return messages[-max_turns * 2:]

def store_message(session_id, role, content):
    id_str = str(uuid.uuid4())
    timestamp = datetime.datetime.utcnow().isoformat()
    embedding = embed_model.encode(content).tolist()
    chat_collection.insert([
        [id_str],
        [session_id],
        [content],
        [role],
        [timestamp],
        [embedding]
    ])
    chat_collection.flush()

def generate_summary(history):
    history_text = "\n".join([f"{role}: {msg}" for role, msg in history])
    prompt = f"Summarize this conversation briefly:\n{history_text}\nSummary:"
    messages = [{"role": "user", "content": prompt}]
    response = llm.create_chat_completion(messages)
    return response["choices"][0]["message"]["content"]


def store_session_summary(session_id, history):
    summary = generate_summary(history)
    embedding = embed_model.encode(summary).tolist()
    summary_collection.insert([
        [str(uuid.uuid4())],  # id
        [session_id],
        [summary],
        [datetime.datetime.utcnow().isoformat()],
        [embedding]
    ])
    summary_collection.flush()


def recall_relevant_summary(user_query):
    embedding = embed_model.encode(user_query).tolist()
    results = summary_collection.search(
        data=[embedding],
        anns_field="summary_embedding",
        param={"metric_type": "COSINE", "params": {"nprobe": 10}},
        limit=1,
        output_fields=["summary", "session_id"]
    )
    if results and results[0]:
        return results[0][0].entity.get("summary")
    return None

# ========== Main Chat Function ==========

def chat_stream(message, history):
    global session_id

    # Try to recall relevant summary
    recalled_summary = recall_relevant_summary(message)
    if recalled_summary:
        memory_context = f"Here is a memory from a past session that may help:\n{recalled_summary}\n\n"
    else:
        memory_context = ""


    user_profile_memory = get_user_profile_memory(session_id)
    if user_profile_memory:
        memory_context += f"Here is what I know about the user:\n{user_profile_memory}\n\n"

    messages = [
    {
        "role": "system",
        "content": memory_context + """
You are a helpful, friendly, and conversational AI companion. Talk in a natural, concise manner. Be human-like: use expressions like "Yeah...", "Okay...", "I get that", "I understand".
Show personality with phrases like "Oh wow!", "No way!", "Seriously?".
Never sound robotic or like customer service.
"""
    }
]

    for user, bot in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": bot})

    messages.append({"role": "user", "content": message})
    store_message(session_id, "user", message)


    extract_and_store_profile_info(message, session_id, profile_collection, embed_model)


    # Stage 1: Thinking/Inner Monologue
    thinking_prompt = messages + [{"role": "user", "content": "Think through your reasoning before responding. What should you consider before replying to the user?"}]
    thinking_response = llm.create_chat_completion(thinking_prompt)
    thinking_text = thinking_response["choices"][0]["message"]["content"]

    # Optionally yield the "thinking" phase (e.g., for logging or debugging)
    print(f"\n[Thinking Phase]:\n{thinking_text}\n")

    # Stage 2: Final Response using internal reflection as memory
    final_prompt = messages + [
    {"role": "system", "content": f"Here's your thought process: {thinking_text}\nNow generate the final reply."}
    ]

    full_reply = ""
    for chunk in llm.create_chat_completion(truncate_history(final_prompt), stream=True):
      delta = chunk["choices"][0].get("delta", {})
      token = delta.get("content", "")
      full_reply += token
      yield full_reply


    store_message(session_id, "assistant", full_reply)

    # End session and store summary
    if message.lower().strip() in ["end session", "exit", "bye"]:
        store_session_summary(session_id, history + [(message, full_reply)])
        session_id = str(int(time.time()))
        print("Session summarized and stored.")

# # Test before launching
# print("\n=== Running tests ===")
# test_message = "Hi, My name is Sam and I am a University Student. I love reading and coding."
# profile_data = extract_user_profile_info(test_message)
# print("Extracted:", profile_data)
# extract_and_store_profile_info(test_message, session_id, profile_collection, embed_model)
# print("=== Tests complete ===\n")


gr.ChatInterface(
    fn=chat_stream,
    title="LLaMA 3.2 1B Chatbot (with memory recall)",
    description="Conversational assistant with Milvus memory and session summarization"
).launch(share=True, debug=True)

/usr/local/lib/python3.11/dist-packages/gradio/chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fd389c1ce6c5e96d4b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Extracted profile info: {}
Profile saved/updated for session: 1748759333


llama_perf_context_print:        load time =   66531.93 ms
llama_perf_context_print: prompt eval time =   66531.50 ms /   237 tokens (  280.72 ms per token,     3.56 tokens per second)
llama_perf_context_print:        eval time =   72414.78 ms /   129 runs   (  561.35 ms per token,     1.78 tokens per second)
llama_perf_context_print:       total time =  139123.97 ms /   366 tokens
Llama.generate: 204 prefix-match hit, remaining 2 prompt tokens to eval



[Thinking Phase]:
Okay, so the user just said "Hello." I need to respond in a friendly and natural way. First, I should acknowledge their greeting. Maybe say "Hello there!" to keep it warm. Then, I can offer help by asking how I can assist them today. I should make sure my response feels genuine and not too formal. Also, I should keep it concise to maintain a natural flow. Let me put that together: "Hello there! How can I assist you today?" That sounds good. It's friendly, acknowledges their greeting, and opens the door for them to share what they need help with.




...

Extracted profile info: {'name': 'Shakti', 'profession': 'So happy to talk to you'}
Profile saved/updated for session: 1748759333


Llama.generate: 27 prefix-match hit, remaining 279 prompt tokens to eval
llama_perf_context_print:        load time =   66531.93 ms
llama_perf_context_print: prompt eval time =   74902.23 ms /   279 tokens (  268.47 ms per token,     3.72 tokens per second)
llama_perf_context_print:        eval time =   21508.55 ms /    39 runs   (  551.50 ms per token,     1.81 tokens per second)
llama_perf_context_print:       total time =   96463.36 ms /   318 tokens
Llama.generate: 273 prefix-match hit, remaining 2 prompt tokens to eval



[Thinking Phase]:
Hmm, okay, so the user has asked me to think through my reasoning before responding. That's an interesting request. Let me try to figure out how to approach this.



llama_perf_context_print:        load time =   66531.93 ms
llama_perf_context_print: prompt eval time =     920.01 ms /     2 tokens (  460.01 ms per token,     2.17 tokens per second)
llama_perf_context_print:        eval time =   19190.43 ms /    33 runs   (  581.53 ms per token,     1.72 tokens per second)
llama_perf_context_print:       total time =   20189.10 ms /    35 tokens


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fd389c1ce6c5e96d4b.gradio.live
